<a href="https://colab.research.google.com/github/AmirJlr/RecSys/blob/master/LLM-based/02_BookRecLLM_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain langchain-community langchain-google-genai langchain-chroma transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 8.6 MB/s eta 0:00:00
 

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

### Or use python-dotenv

In [ ]:
import pandas as pd

books = pd.read_csv("/content/drive/MyDrive/recommender_systems/LLM/books_cleaned.csv")

In [ ]:
books.tagged_description.to_csv("tagged_description.txt", sep="\n", index=False, header=False)

In [ ]:
raw_documents = TextLoader("tagged_description.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=0, chunk_overlap=0, separator='\n')

documents = text_splitter.split_documents(raw_documents)

Streaming output truncated to the last 5000 lines.


In [ ]:
db_books = Chroma.from_documents(
    documents, embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001")
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
query = "A book about football"

recs = db_books.similarity_search(query, k=3)
recs

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(id='594ef880-123e-4dde-b09a-5cf67cf6bd52', metadata={'source': 'tagged_description.txt'}, page_content='9781573226882 An autobiographical memoir by a humorous British author and obsessed soccer fan captures the intensity of a sports fan who measures his life in seasons rather than years'),
 Document(id='13d47a35-6d29-4cab-882f-bcf7087df586', metadata={'source': 'tagged_description.txt'}, page_content="9781406934854 This is a reproduction of the original artefact. Generally these books are created from careful scans of the original. This allows us to preserve the book accurately and present it in the way the author intended. Since the original versions are generally quite old, there may occasionally be certain imperfections within these reproductions. We're happy to make these classics available again for future generations to enjoy!"),
 Document(id='9f69f725-95bd-4652-8be6-07a36ab1ebeb', metadata={'source': 'tagged_description.txt'}, page_content="9781406904833 This is a repr

In [ ]:
recs[2]

Document(id='9f69f725-95bd-4652-8be6-07a36ab1ebeb', metadata={'source': 'tagged_description.txt'}, page_content="9781406904833 This is a reproduction of the original artefact. Generally these books are created from careful scans of the original. This allows us to preserve the book accurately and present it in the way the author intended. Since the original versions are generally quite old, there may occasionally be certain imperfections within these reproductions. We're happy to make these classics available again for future generations to enjoy!")

In [ ]:
books.head(1)

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...


In [ ]:
books[books['isbn13'] == int(recs[0].page_content.split(" ")[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
4700,9781573226882,1573226882,Fever Pitch,Nick Hornby,Biography & Autobiography,http://books.google.com/books/content?id=v5TO8...,An autobiographical memoir by a humorous Briti...,1998.0,3.73,247.0,27781.0,Fever Pitch,9781573226882 An autobiographical memoir by a ...


In [ ]:
def retrieve_semantic_recommendation(dataframe, db, query, num_recs):
    recs = db.similarity_search(query, k=num_recs)

    book_list = []
    for i in range(len(recs)):
        book_list.append(int(recs[i].page_content.strip('"').split(" ")[0]))

    return dataframe[dataframe["isbn13"].isin(book_list)]

In [ ]:
retrieve_semantic_recommendation(books, db_books, "A book about football", 5)

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
4461,9781406904833,140690483X,Nöddebo Parsonage,Bentley,NaN,http://books.google.com/books/content?id=OFJyy...,This is a reproduction of the original artefac...,2019.0,4.32,224.0,8206.0,Nöddebo Parsonage: A Story of Country Life in ...,9781406904833 This is a reproduction of the or...
4463,9781406922905,1406922900,"The Pronouncing Reading Book for Children, Wit...",William L Robinson,History,http://books.google.com/books/content?id=gyiJy...,This is a reproduction of the original artefac...,2019.0,3.66,92.0,2477.0,"The Pronouncing Reading Book for Children, Wit...",9781406922905 This is a reproduction of the or...
4464,9781406934854,1406934852,A Commentary Upon the Gospel According to S. Luke,Saint Cyril (patriarch of Alexandria),History,http://books.google.com/books/content?id=Mclsy...,This is a reproduction of the original artefac...,2019.0,3.59,362.0,669.0,A Commentary Upon the Gospel According to S. Luke,9781406934854 This is a reproduction of the or...
4466,9781406957242,1406957240,The Story of the Life of Lafayette,Mrs John Farrar,History,http://books.google.com/books/content?id=oVd-y...,This is a reproduction of the original artefac...,2019.0,3.93,368.0,29.0,The Story of the Life of Lafayette: As Told by...,9781406957242 This is a reproduction of the or...
4700,9781573226882,1573226882,Fever Pitch,Nick Hornby,Biography & Autobiography,http://books.google.com/books/content?id=v5TO8...,An autobiographical memoir by a humorous Briti...,1998.0,3.73,247.0,27781.0,Fever Pitch,9781573226882 An autobiographical memoir by a ...
